# 01 — Synthetic Training Data Generation

Generates multi-turn Romanian conversations for fine-tuning the city recommendation chatbot.

## Design
- **Phase 1 (turns 1–3):** Assistant asks clarifying questions about preferences (no city facts).
- **Phase 2 (final turn):** A `[CONTEXT RAG]` system message injects city data; assistant presents ONLY that.

The model will never be trained to memorise city facts — it learns *conversation behaviour* and *grounded presentation*.

## Resumable
Re-run this notebook as many times as needed when API limits reset.  
Completed scenarios are skipped automatically. Data is appended to `data/raw_conversations.jsonl`.

## Prerequisites
```bash
pip install openai
```
Get a **free** Groq API key at https://console.groq.com/keys then set it:
```bash
set GROQ_API_KEY=gsk_...
```
Alternative backends: `"gemini"` (set `GEMINI_API_KEY`), `"anthropic"` (set `ANTHROPIC_API_KEY`), `"openai"` (set `OPENAI_API_KEY`).

### Groq free-tier rate limits
- **llama-3.3-70b-versatile** (default): 30 req/min, ~6 000 TPM — the loop sleeps **12 s** between calls.
- **llama-3.1-8b-instant**: higher TPM allowance, faster but lower quality — change `MODEL` in Step 2.

In [1]:
import sys
import os
from pathlib import Path

# Add repo root to path so city_preprocessing and fine_tuning imports work
REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FINE_TUNING_DIR = REPO_ROOT / "fine_tuning"
CITIES_JSON_DIR = REPO_ROOT / "informations" / "cities_json"
CITY_INDEX_PATH = FINE_TUNING_DIR / "configs" / "city_index.json"

print(f"Repo root : {REPO_ROOT}")
print(f"Cities JSON: {CITIES_JSON_DIR} — exists: {CITIES_JSON_DIR.exists()}")

Repo root : C:\Facultate\UnderstandingLLms\Proiect\repo\llms
Cities JSON: C:\Facultate\UnderstandingLLms\Proiect\repo\llms\informations\cities_json — exists: True


## Step 1 — Build city index (run once)

In [2]:
from fine_tuning.src.city_loader import load_city_index, save_city_index

if CITY_INDEX_PATH.exists():
    print(f"City index already exists at {CITY_INDEX_PATH} — skipping rebuild.")
    print("Delete the file and re-run this cell to force a rebuild.")
else:
    print("Building city index from JSON files...")
    city_index = load_city_index(CITIES_JSON_DIR)
    save_city_index(city_index, CITY_INDEX_PATH)
    print(f"Saved {len(city_index)} cities to {CITY_INDEX_PATH}")

City index already exists at C:\Facultate\UnderstandingLLms\Proiect\repo\llms\fine_tuning\configs\city_index.json — skipping rebuild.
Delete the file and re-run this cell to force a rebuild.


In [3]:
from fine_tuning.src.city_loader import load_saved_city_index

city_index = load_saved_city_index(CITY_INDEX_PATH)
print(f"Loaded city index: {len(city_index)} cities")

# Quick sanity check
sample = city_index.get("brașov") or next(iter(city_index.values()))
print(f"\nSample entry: {sample['display_name']}")
print(f"  Sections available: {sample['section_titles']}")

Loaded city index: 315 cities

Sample entry: Brașov
  Sections available: ['Introducere', 'Geografie', 'Demografie', 'Educație', 'Economie', 'Turism', 'Obiective turistice']


## Step 2 — Configure API client

In [4]:
import os
from fine_tuning.src.api_client import make_client

# ---- Configuration ----
BACKEND = "groq"      # "groq" (default) | "gemini" | "anthropic" | "openai"
API_KEY = "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"
# MODEL = "llama-3.1-8b-instant"  # uncomment for faster/higher-TPM alternative

client = make_client(backend=BACKEND, api_key=API_KEY)

# Each backend exposes its recommended inter-request sleep via .inter_request_sleep
BACKEND_SLEEP = client.inter_request_sleep

print(f"Backend  : {BACKEND}")
print(f"Model    : {client.model}")
print(f"Sleep    : {BACKEND_SLEEP}s between requests")

Backend  : groq
Model    : llama-3.3-70b-versatile
Sleep    : 12.0s between requests


## Step 3 — Generate scenarios for this run

In [5]:
from fine_tuning.src.progress_tracker import ProgressTracker
from fine_tuning.src.scenario_generator import generate_scenarios

VARIANTS_PER_PERSONA = 6   # 12 personas × 6 variants = 72 scenarios per run

tracker = ProgressTracker()
run_id  = tracker.start_run()
print(f"Run ID: {run_id}")
print(tracker.summary())

all_scenarios  = generate_scenarios(city_index, run_id, variants_per_persona=VARIANTS_PER_PERSONA)
pending        = [s for s in all_scenarios if not tracker.is_done(s["scenario_id"]) and not tracker.is_exhausted(s["scenario_id"])]

print(f"\nTotal scenarios this run : {len(all_scenarios)}")
print(f"Already completed (skip) : {len(all_scenarios) - len(pending)}")
print(f"Pending (to generate)    : {len(pending)}")

Run ID: run_a892a0a1
Completed: 71 | Permanently failed: 0 | Runs: 10

Total scenarios this run : 72
Already completed (skip) : 0
Pending (to generate)    : 72


## Step 4 — Run generation loop

In [6]:
import time
from fine_tuning.src.city_loader import build_city_card
from fine_tuning.src.conversation_validator import validate_conversation
from fine_tuning.configs.generation_config import (
    GENERATION_SYSTEM_PROMPT,
    GENERATION_USER_PROMPT_TEMPLATE,
)


def build_user_prompt(scenario: dict) -> str:
    """Render the per-scenario user prompt from the template."""
    pref = scenario["preference_vector"]
    preference_summary = "\n".join(
        f"  - {k}: {v}" for k, v in pref.items() if not k.startswith("_")
    )
    city_cards = "\n\n".join(
        build_city_card(city) for city in scenario["cities"]
    )
    return GENERATION_USER_PROMPT_TEMPLATE.format(
        persona_description=scenario["persona_description"],
        preference_summary=preference_summary,
        city_cards=city_cards,
    )


success_count = 0
fail_count    = 0

for idx, scenario in enumerate(pending, start=1):
    sid = scenario["scenario_id"]
    print(f"[{idx}/{len(pending)}] {scenario['persona_type']} v{scenario['variant_idx']} — {sid[:40]}")

    try:
        user_prompt = build_user_prompt(scenario)
        raw_result  = client.generate(GENERATION_SYSTEM_PROMPT, user_prompt)

        # Validate before recording
        warnings = validate_conversation(raw_result, city_index)
        if warnings:
            print(f"  Warnings: {warnings}")

        # Attach scenario metadata
        raw_result.setdefault("metadata", {})
        raw_result["metadata"]["scenario_id"]  = sid
        raw_result["metadata"]["persona_type"] = scenario["persona_type"]
        raw_result["metadata"]["run_id"]        = run_id
        raw_result["metadata"]["warnings"]      = warnings

        tracker.record_success(sid, raw_result, run_id)
        success_count += 1
        print(f"  OK — turns: {raw_result['metadata'].get('turn_count', '?')}")

    except Exception as exc:
        print(f"  FAILED: {exc}")
        tracker.record_failure(sid, run_id, error=str(exc))
        fail_count += 1

    # Respect backend-specific rate limit
    time.sleep(BACKEND_SLEEP)

tracker.finish_run(run_id)
print(f"\nRun {run_id} complete — success: {success_count}, failed: {fail_count}")
print(tracker.summary())

[1/72] lucrator_remote v3 — lucrator_remote::6e4a18e8::run_a892a0a1
  OK — turns: 7
[2/72] lucrator_industrie v1 — lucrator_industrie::5c6a90dd::run_a892a0
  OK — turns: 7
[3/72] lucrator_remote v1 — lucrator_remote::52c34cfb::run_a892a0a1
  Warnings: ['high_turn_count:8', 'possible_hallucination:Deta']
  OK — turns: 7
[4/72] refugiat_intern v5 — refugiat_intern::627590f0::run_a892a0a1
  OK — turns: 7
[5/72] familie_cu_copii v0 — familie_cu_copii::eb9876d6::run_a892a0a1
  Warnings: ['high_turn_count:8', 'possible_hallucination:Titu']
  OK — turns: 7
[6/72] refugiat_intern v1 — refugiat_intern::9c68ba04::run_a892a0a1
  OK — turns: 7
[7/72] familie_cu_copii v4 — familie_cu_copii::0818898f::run_a892a0a1
  OK — turns: 7
[8/72] tanar_profesionist_it v0 — tanar_profesionist_it::230e37a1::run_a89
  Warnings: ['high_turn_count:8', 'possible_hallucination:București']
  OK — turns: 7
[9/72] refugiat_intern v0 — refugiat_intern::955a0184::run_a892a0a1
  OK — turns: 7
[10/72] lucrator_remote v5 — 

KeyboardInterrupt: 

## Step 5 — Quick quality check

In [ ]:
import json
from fine_tuning.src.progress_tracker import RAW_CONVERSATIONS_FILE

lines = RAW_CONVERSATIONS_FILE.read_text(encoding="utf-8").splitlines()
conversations = [json.loads(l) for l in lines if l.strip()]
print(f"Total conversations in dataset: {len(conversations)}")

# Print a sample
if conversations:
    sample = conversations[-1]
    print(f"\n--- Latest conversation ({sample['metadata'].get('persona_type')}) ---")
    for msg in sample.get("conversation", []):
        role    = msg["role"].upper()
        preview = msg["content"][:200].replace("\n", " ")
        print(f"  [{role}] {preview}...")